# CFD Surrogate Model — U-Net Training

This notebook trains a **U-Net** neural network to predict 2-D velocity fields
from building-geometry inputs. It picks up where `data_preprocessing_Latest.ipynb`
left off: that notebook saved preprocessed `.npy` files to `/content/cfd_workshop/processed/`.

> **Before running this notebook** make sure the preprocessing notebook has been executed
> in the **same Colab session** (local `/content/` storage is wiped between sessions).

## Notebook overview

| Section | Topic |
|:-------:|-------|
| 1 | Environment setup — Mount Drive, restore processed data, imports, CONFIG |
| 2 | Data loading — Dataset class, DataLoaders, sample visualization |
| 3 | U-Net architecture — encoder, bottleneck, decoder, skip connections |
| 4 | Training — MSE loss, Adam optimizer, LR scheduling |
| 5 | Loss curves — training vs validation MSE |
| 6 | Evaluation — per-case RMSE, visual comparisons, scatter plot |
| 7 | Save results to Google Drive |


In [ ]:
# Mount Google Drive so we can save the trained model and plots.
# Running this cell will open a link asking you to authorize Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted.')
except Exception as e:
    IN_COLAB = False
    print(f'Not running in Colab ({type(e).__name__}). Paths use current directory.')


In [ ]:
# ── Restore processed data from Google Drive to local Colab storage ───────────
#
# WHY THIS CELL EXISTS
# --------------------
# The preprocessing notebook writes processed .npy files to /content/cfd_workshop/
# which is fast LOCAL storage tied to that specific Colab session.
# When you open this training notebook in a NEW session, that storage is gone.
#
# SOLUTION
# --------
# The preprocessing notebook's final cell saved everything to Google Drive.
# This cell copies it back to local storage once at startup so all subsequent
# DataLoader reads use fast local I/O — not slow Drive I/O every epoch.
#
# WHAT IT DOES
# ------------
# 1. If processed data already exists locally (preprocessing ran in THIS session)
#    → skips the copy (nothing to do).
# 2. If processed data is missing locally but exists on Drive
#    → copies it from Drive to /content/cfd_workshop/ (one-time cost, then fast).
# 3. If neither exists → raises a clear error explaining what to do.

import shutil
from pathlib import Path

# This must match the path used in the preprocessing notebook's save cell
DRIVE_BACKUP    = Path('/content/drive/MyDrive/cfd_workshop_backup')
LOCAL_WORKSPACE = Path('/content/cfd_workshop')
LOCAL_PROCESSED = LOCAL_WORKSPACE / 'processed'

if LOCAL_PROCESSED.exists():
    # Data is already local — preprocessing ran in this same session
    n = sum(1 for _ in LOCAL_PROCESSED.rglob('*.npy'))
    print(f'Processed data already available locally ({n} .npy files). No restore needed.')

elif DRIVE_BACKUP.exists():
    # Data is on Drive — copy it back to local storage
    print('Processed data not found locally. Restoring from Google Drive...')
    print(f'  Source : {DRIVE_BACKUP}')
    print(f'  Dest   : {LOCAL_WORKSPACE}')
    print('  (This one-time copy takes ~1-2 minutes depending on dataset size)')

    shutil.copytree(
        src=str(DRIVE_BACKUP),
        dst=str(LOCAL_WORKSPACE),
        dirs_exist_ok=True   # safe to run even if /content/cfd_workshop/ partially exists
    )

    n = sum(1 for _ in LOCAL_PROCESSED.rglob('*.npy'))
    print(f'\nRestore complete — {n} .npy files ready at {LOCAL_PROCESSED}')
    print('All subsequent data loading will use fast local storage.')

else:
    # Neither local nor Drive — preprocessing has not been run
    raise FileNotFoundError(
        '\n'
        'Processed data not found in either location:\n'
        f'  Local : {LOCAL_PROCESSED}\n'
        f'  Drive : {DRIVE_BACKUP}\n'
        '\n'
        'To fix this:\n'
        '  1. Open data_preprocessing_Latest.ipynb in a new Colab session.\n'
        '  2. Run all cells including the final Save-to-Drive cell.\n'
        '  3. Come back to this notebook and re-run from the top.'
    )


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import random
from pathlib import Path

# ── Scientific computing ───────────────────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── PyTorch — the deep-learning framework we use to build and train the U-Net ──
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION DICTIONARY
# All important settings live here.
# To experiment: change values in this cell only — don't edit code elsewhere.
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    # Where the preprocessing notebook saved its output
    'processed_dir': Path('/content/cfd_workshop/processed'),
    # Where to save model checkpoints and statistics during this run
    'models_dir':    Path('/content/cfd_workshop/models'),
    # Where to save evaluation plots
    'outputs_dir':   Path('/content/cfd_workshop/outputs'),
    # Google Drive destination for final results (edit to match your Drive layout)
    'drive_dir':     Path('/content/drive/MyDrive/cfd_workshop_results'),

    # Google Drive folder where the preprocessing notebook saved the processed data.
    # IMPORTANT: this must match the path used in the preprocessing notebook's
    # 'Save to Drive' cell. Edit here if you chose a different folder name.
    'drive_backup':  Path('/content/drive/MyDrive/cfd_workshop_backup'),

    # ── U-Net architecture ─────────────────────────────────────────────────────
    # The preprocessing notebook produced 3 channels: SDF, inlet velocity, region labels
    'in_channels':  3,
    'out_channels': 1,           # we predict one field: velocity magnitude
    # Feature-map widths at each encoder depth level.
    # Deeper = more capacity but also more memory and slower training.
    'features':     [64, 128, 256, 512, 1024],

    # ── Training hyperparameters ───────────────────────────────────────────────
    'batch_size':   8,           # samples processed together per gradient step
    'epochs':       20,          # total training passes over the dataset
    'lr':           1e-4,        # initial Adam learning rate
    'lr_patience':  5,           # wait this many epochs before reducing LR
    'lr_factor':    0.5,         # multiply LR by this when reducing (0.5 = halve it)
    'weight_decay': 1e-5,        # L2 regularisation — penalises very large weights

    # ── Other ──────────────────────────────────────────────────────────────────
    'img_size': 128,             # spatial grid size (must match preprocessing)
    'seed':     42,              # fixed seed for reproducibility
}

# Derived paths (depend on other CONFIG values, so defined separately)
CONFIG['checkpoint_path'] = CONFIG['models_dir'] / 'unet_best.pth'

# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# Fixing all random seeds ensures that running the notebook twice gives the
# same result — essential for comparing experiments fairly.
# ─────────────────────────────────────────────────────────────────────────────
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# Neural networks train much faster on a GPU.
# Colab: Runtime → Change runtime type → GPU to get a free T4/A100.
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True   # auto-tune convolutions for this GPU
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')
else:
    print('No GPU found. Training will be slow — consider enabling a GPU runtime.')

# Create output directories (parents=True creates any missing parent folders)
for key in ('models_dir', 'outputs_dir'):
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print('\nConfiguration ready.')
for k, v in CONFIG.items():
    print(f'  {k:<20}: {v}')


In [ ]:
# Verify that the preprocessed dataset is present before we try to train.
# If the preprocessing notebook was not run in this session, the paths won't exist.
print('Checking processed dataset structure...')
print()

all_ok = True
for split in ['train', 'val', 'test']:
    inp_dir = CONFIG['processed_dir'] / split / 'inputs'
    tgt_dir = CONFIG['processed_dir'] / split / 'targets'

    if not inp_dir.exists():
        print(f'  ERROR: {inp_dir} not found.')
        all_ok = False
        continue

    n_inp = len(list(inp_dir.glob('*.npy')))
    n_tgt = len(list(tgt_dir.glob('*.npy')))
    print(f'  {split:>5}/  inputs={n_inp:>4}  targets={n_tgt:>4}')

if not all_ok:
    raise FileNotFoundError(
        'Processed data not found. '
        'Please run data_preprocessing_Latest.ipynb first in this session.'
    )

# Quick shape check on one training file
sample_inp = next(
    f for f in (CONFIG['processed_dir'] / 'train' / 'inputs').glob('*.npy')
    if 'flip' not in f.stem
)
sample_tgt = (
    CONFIG['processed_dir'] / 'train' / 'targets' / (sample_inp.stem + '.npy')
)
X_chk = np.load(sample_inp)
y_chk = np.load(sample_tgt)

print()
print(f'Sample input  shape : {X_chk.shape}  dtype : {X_chk.dtype}')
print(f'Sample target shape : {y_chk.shape}  dtype : {y_chk.dtype}')
print(f'Input  value range  : [{X_chk.min():.3f}, {X_chk.max():.3f}]  (already scaled by preprocessing)')
print(f'Target value range  : [{y_chk.min():.3f}, {y_chk.max():.3f}] m/s  (raw velocity)')
print()
print('Data structure verified — ready for training.')


---
# Section 2 — Data Loading

The preprocessing notebook already did the hard work:

- **Inputs** are saved as `(3, 128, 128)` float32 tensors with three physics-aware channels:
  - Ch 0: Signed Distance Function (SDF), standardized (z-score)
  - Ch 1: Inlet velocity, standardized (z-score)
  - Ch 2: Flow-region labels, min-max normalized to [0, 1]
- **Targets** are saved as `(128, 128)` float32 arrays of raw velocity magnitude (m/s).

Since inputs are **already scaled**, our `CFDDataset` class just loads them.
We do need to normalize the targets to [0, 1] for stable training; we compute
the min/max statistics from the **training set only** (using val/test statistics
here would be "data leakage").


In [ ]:
# Redefine the custom velocity colormap from the preprocessing notebook.
# Using the same colormap everywhere keeps all velocity-field plots comparable.
# 25 colors span 0 → 13 m/s with equal-width steps.

HEX_COLORS = [
    '#0000ff', '#0068ff', '#0093ff', '#00b4ff', '#00d0ff',
    '#00e9ff', '#00ffff', '#00ffe9', '#00ffd0', '#00ffb4',
    '#00ff93', '#00ff68', '#00ff00', '#68ff00', '#93ff00',
    '#b4ff00', '#d0ff00', '#e9ff00', '#ffff00', '#ffe900',
    '#ffd000', '#ffb400', '#ff9300', '#ff6800', '#ff0000',
]

VELOCITY_CMAP = mcolors.ListedColormap(HEX_COLORS)
VELOCITY_BOUNDARIES = [
    0.0, 0.52, 1.04, 1.56, 2.08, 2.6,  3.12, 3.64, 4.16, 4.68,
    5.2, 5.72, 6.24, 6.76, 7.28, 7.8,  8.32, 8.84, 9.36, 9.88,
    10.4, 10.92, 11.44, 11.96, 12.48, 13.0,
]
VELOCITY_NORM = mcolors.BoundaryNorm(VELOCITY_BOUNDARIES, VELOCITY_CMAP.N)
print('Velocity colormap ready: 25 colours, 0-13 m/s.')


In [ ]:
# Compute target (velocity) normalization statistics from the TRAINING set.
# We scan every training target file to find the global min and max velocity.
# These values are then used to scale targets to [0, 1] before computing loss.
#
# Why normalize targets?
#   Raw velocities range ~0-13 m/s. While MSE can work on raw values, keeping
#   inputs and outputs on a similar scale (~[0,1]) generally leads to more
#   stable gradients and faster convergence.
#
# Why training set only?
#   If we used val/test targets to compute min/max, those splits would
#   indirectly influence training — a subtle form of data leakage.

print('Scanning training targets to compute min/max statistics...')

tgt_dir   = CONFIG['processed_dir'] / 'train' / 'targets'
tgt_files = sorted(tgt_dir.glob('*.npy'))
n_files   = len(tgt_files)

t_min =  float('inf')
t_max = -float('inf')

for i, fp in enumerate(tgt_files, 1):
    arr   = np.load(fp)
    t_min = min(t_min, float(arr.min()))
    t_max = max(t_max, float(arr.max()))
    if i % 200 == 0 or i == n_files:
        print(f'  [{i:>4}/{n_files}]  running min={t_min:.4f}  max={t_max:.4f}')

print()
print(f'Target statistics (from {n_files} training files):')
print(f'  min = {t_min:.6f} m/s')
print(f'  max = {t_max:.6f} m/s')

# Save so we can reload them for inference without re-scanning
stats_file = CONFIG['models_dir'] / 'target_stats.json'
with open(stats_file, 'w') as f:
    json.dump({'min': t_min, 'max': t_max}, f, indent=2)
print(f'\nSaved target statistics → {stats_file}')

# Make these accessible as module-level constants for the Dataset class
TARGET_MIN = t_min
TARGET_MAX = t_max


In [ ]:
class CFDDataset(Dataset):
    """
    PyTorch Dataset that loads preprocessed CFD data for one split.

    Folder layout expected:
        processed/<split>/inputs/  →  case_<id>.npy  shape (3, 128, 128)
        processed/<split>/targets/ →  case_<id>.npy  shape (128, 128)

    The Dataset:
      1. Loads the input tensor as-is (already scaled by the preprocessing notebook).
      2. Adds a channel dimension to the target: (128, 128) → (1, 128, 128).
      3. Normalises the target velocity to [0, 1] using TARGET_MIN / TARGET_MAX.

    PyTorch's DataLoader calls __getitem__(idx) repeatedly to assemble batches.
    """

    def __init__(self, split: str):
        """
        Args:
            split: one of 'train', 'val', or 'test'
        """
        self.input_dir  = CONFIG['processed_dir'] / split / 'inputs'
        self.target_dir = CONFIG['processed_dir'] / split / 'targets'

        # Collect every .npy file and sort for reproducibility
        all_files   = sorted(self.input_dir.glob('*.npy'))
        self.stems  = [f.stem for f in all_files]   # e.g. 'case_42', 'case_42_hflip'

        # Extract the integer case ID for use in evaluation labelling
        # 'case_42'       → 42
        # 'case_42_hflip' → 42  (same geometry, flipped augmentation)
        self.case_ids = [int(s.split('_')[1]) for s in self.stems]

    def __len__(self) -> int:
        """Total number of samples in this split (including augmented copies)."""
        return len(self.stems)

    def __getitem__(self, idx: int):
        """
        Return one (input, target) pair.

        Returns
        -------
        X : torch.FloatTensor  shape (3, 128, 128)
            Three scaled input channels (SDF / inlet-vel / region).
        y : torch.FloatTensor  shape (1, 128, 128)
            Velocity magnitude normalised to [0, 1].
        """
        stem = self.stems[idx]

        # Load the scaled input tensor produced by the preprocessing notebook.
        X = np.load(self.input_dir  / f'{stem}.npy').copy()  # (3, 128, 128)

        # Load the raw velocity target and add a channel dimension so that
        # both X and y have a leading channel axis: (C, H, W).
        y = np.load(self.target_dir / f'{stem}.npy')          # (128, 128)
        y = y[np.newaxis, ...]                                # (1, 128, 128)

        # Normalise target to [0, 1] using training-set statistics.
        # The small epsilon (1e-8) prevents division by zero in edge cases.
        y = (y - TARGET_MIN) / (TARGET_MAX - TARGET_MIN + 1e-8)

        # Convert numpy arrays to PyTorch float tensors
        return torch.from_numpy(X).float(), torch.from_numpy(y).float()


print('CFDDataset class defined.')


In [ ]:
# Create Dataset objects for all three splits
train_dataset = CFDDataset('train')
val_dataset   = CFDDataset('val')
test_dataset  = CFDDataset('test')

# DataLoader wraps a Dataset and provides:
#   - Automatic batching (batch_size samples per iteration)
#   - Shuffling of training data each epoch (important: prevents the model
#     from memorising the order of samples)
#   - Parallel data loading (num_workers > 0 is faster locally, but
#     num_workers=0 is safer inside Colab to avoid multiprocessing issues)
#   - pin_memory=True: pre-loads data into pinned (page-locked) CPU memory
#     which speeds up the CPU→GPU transfer step

_pin = (DEVICE.type == 'cuda')

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=0, pin_memory=_pin)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=_pin)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=_pin)

print('DataLoaders ready.')
print()
print(f'  Train : {len(train_dataset):>4} samples  |  {len(train_loader):>3} batches')
print(f'  Val   : {len(val_dataset):>4} samples  |  {len(val_loader):>3} batches')
print(f'  Test  : {len(test_dataset):>4} samples  |  {len(test_loader):>3} batches')

# Verify shapes by loading one sample
X0, y0 = train_dataset[0]
print()
print(f'Input  tensor shape : {tuple(X0.shape)}   dtype : {X0.dtype}')
print(f'Target tensor shape : {tuple(y0.shape)}   dtype : {y0.dtype}')
print(f'Input  value range  : [{X0.min():.3f}, {X0.max():.3f}]')
print(f'Target value range  : [{y0.min():.3f}, {y0.max():.3f}]  (normalised to [0,1])')


In [ ]:
# Visualise 3 random training samples before training starts.
# This is a sanity check: confirm that inputs look correct and targets
# match the expected velocity-field patterns.
n_show     = 3
viz_idx    = random.sample(range(len(train_dataset)), n_show)
ch_names   = ['SDF (standardized)', 'Inlet velocity (standardized)', 'Region labels (normalized)']
ch_cmaps   = ['RdBu_r', 'viridis', 'tab10']

fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))

for row, idx in enumerate(viz_idx):
    X_v, y_v = train_dataset[idx]
    cid       = train_dataset.case_ids[idx]

    # Plot the three input channels
    for col in range(3):
        im = axes[row, col].imshow(
            X_v[col].numpy(), origin='lower', cmap=ch_cmaps[col]
        )
        axes[row, col].set_title(f'Ch {col+1}: {ch_names[col]}', fontsize=9)
        axes[row, col].axis('off')
        plt.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)

    # Denormalise target back to m/s for display
    y_phys = y_v.numpy()[0] * (TARGET_MAX - TARGET_MIN) + TARGET_MIN
    im = axes[row, 3].imshow(
        y_phys, origin='lower', cmap=VELOCITY_CMAP, norm=VELOCITY_NORM
    )
    axes[row, 3].set_title(f'Target velocity (m/s)  [case {cid}]', fontsize=9)
    axes[row, 3].axis('off')
    plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04, label='m/s')

fig.suptitle(
    'Training samples: input channels (cols 1-3) and velocity target (col 4)',
    fontsize=12, y=1.01
)
plt.tight_layout()
out_path = str(CONFIG['outputs_dir'] / 'training_samples.png')
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

---
# Section 3 — U-Net Architecture



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILDING BLOCK 1: DoubleConv
# Two convolutional layers with batch normalisation and a residual shortcut.
# ─────────────────────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    """
    Two Conv2d(3×3) → BatchNorm2d → ReLU layers with a residual shortcut.

    Why two convolutions?
        A single convolution has a limited "receptive field" — it can only look
        at a small neighbourhood at once. Two convolutions expand the receptive
        field so each output pixel incorporates information from a larger region.

    Why BatchNorm?
        Batch Normalisation rescales each feature map to have zero mean and unit
        variance across the batch. This prevents any single feature from
        dominating and makes training much more stable.

    Why the residual shortcut?
        Adding the input directly to the output (the "skip" inside this block)
        creates an identity path for gradients. Without it, deep networks
        suffer from vanishing gradients — the signal becomes too weak to update
        early layers. With it, the block only needs to learn the *difference*
        from the identity, which is easier.
    """

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()

        # Main path: two Conv+BN+ReLU layers
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

        # Shortcut path: if the number of channels changes, use a 1×1 conv to
        # match dimensions; otherwise use Identity (pass through unchanged).
        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
            if in_ch != out_ch else nn.Identity()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Add the residual shortcut to the main path, then apply ReLU
        return self.relu(self.net(x) + self.shortcut(x))


# ─────────────────────────────────────────────────────────────────────────────
# BUILDING BLOCK 2: EncoderBlock
# One DoubleConv (produces the skip feature), then strided convolution to halve
# the spatial resolution (replaces the traditional MaxPool).
# ─────────────────────────────────────────────────────────────────────────────
class EncoderBlock(nn.Module):
    """
    Encoder step: extract features at the current resolution, then downsample.

    We use a strided convolution (stride=2) instead of MaxPool for downsampling.
    MaxPool just discards values; a strided convolution *learns* how to downsample,
    which can preserve more useful information.
    """

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        
        # stride=2 → each output pixel covers a 2×2 input region → spatial size halved
        self.down = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=2, padding=1, bias=False)

    def forward(self, x):
        skip   = self.conv(x)    # full-resolution features (sent to decoder via skip conn)
        x_down = self.down(skip) # downsampled features (continue through encoder)
        return x_down, skip


# ─────────────────────────────────────────────────────────────────────────────
# BUILDING BLOCK 3: Bottleneck
# The deepest part of the network — smallest spatial size, most feature maps.
# Captures the most abstract global patterns (overall jet direction, wake size).
# ─────────────────────────────────────────────────────────────────────────────
class Bottleneck(nn.Module):
    """DoubleConv at the lowest spatial resolution. No further downsampling."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(x)


# ─────────────────────────────────────────────────────────────────────────────
# BUILDING BLOCK 4: DecoderBlock
# Upsample → reduce channels → concatenate the matching encoder skip → DoubleConv.
# The skip connection re-introduces the fine spatial detail that was compressed away.
# ─────────────────────────────────────────────────────────────────────────────
class DecoderBlock(nn.Module):
    """
    Decoder step: upsample, fuse skip connection, refine features.

    Steps:
      1. Bilinear upsampling × 2  →  spatial size doubles.
      2. 1×1 conv to halve channels  →  reduces parameters before concatenation.
      3. Concatenate the matching encoder skip  →  recovers fine spatial detail.
      4. DoubleConv to blend upsampled + skip features into a coherent map.
    """

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # Bilinear upsampling is smooth and parameter-free; align_corners=False
        # is the recommended setting to avoid subtle grid alignment artefacts.
        self.up     = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        # Reduce channels before concatenation (in_ch → out_ch)
        self.reduce = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        # After concat: out_ch (upsampled) + out_ch (skip) = out_ch * 2
        self.conv   = DoubleConv(out_ch * 2, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        x = self.reduce(x)
        x = torch.cat([skip, x], dim=1)   # concatenate along the channel axis
        return self.conv(x)


print('DoubleConv, EncoderBlock, Bottleneck, DecoderBlock defined.')

In [ ]:
class VanillaUNet(nn.Module):
    """
    Complete U-Net with four encoder levels, a bottleneck, and four decoder levels.

    Input  : (batch, in_channels,  128, 128)
    Output : (batch, out_channels, 128, 128)

    Skip connections at each encoder level preserve spatial resolution information
    and allow the decoder to place velocity features at the correct locations.
    """

    def __init__(self,
                 in_channels:  int  = CONFIG['in_channels'],
                 out_channels: int  = CONFIG['out_channels'],
                 features:     list = None):
        super().__init__()

        # Use CONFIG['features'] as default; allow override for testing
        f = features if features is not None else CONFIG['features']

        # ── Encoder (contracting path) ─────────────────────────────────────────
        # Each EncoderBlock: DoubleConv → skip  +  strided-Conv downsample
        self.enc1 = EncoderBlock(in_channels, f[0])  # (3,128,128) → skip(64,128,128) + (64,64,64)
        self.enc2 = EncoderBlock(f[0], f[1])          # (64,64,64)  → skip(128,64,64)  + (128,32,32)
        self.enc3 = EncoderBlock(f[1], f[2])          # (128,32,32) → skip(256,32,32)  + (256,16,16)
        self.enc4 = EncoderBlock(f[2], f[3])          # (256,16,16) → skip(512,16,16)  + (512,8,8)

        # ── Bottleneck (most compressed representation) ────────────────────────
        self.bottleneck = Bottleneck(f[3], f[3] * 2)  # (512,8,8) → (1024,8,8)

        # ── Decoder (expanding path) ───────────────────────────────────────────
        # Each DecoderBlock: upsample + concat(skip) + DoubleConv
        self.dec4 = DecoderBlock(f[3] * 2, f[3])  # (1024,8,8)   + skip4(512,16,16) → (512,16,16)
        self.dec3 = DecoderBlock(f[3], f[2])       # (512,16,16)  + skip3(256,32,32) → (256,32,32)
        self.dec2 = DecoderBlock(f[2], f[1])       # (256,32,32)  + skip2(128,64,64) → (128,64,64)
        self.dec1 = DecoderBlock(f[1], f[0])       # (128,64,64)  + skip1(64,128,128)→ (64,128,128)

        # ── Output head ────────────────────────────────────────────────────────
        # 1×1 convolution maps from feature space to the prediction space.
        # No activation here — the loss function is MSE on normalised values so
        # the output can be any real number.
        self.head = nn.Conv2d(f[0], out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder: each step returns (downsampled_features, skip_features)
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder: each step receives the upsampled features and the skip
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)

        return self.head(x)


# ── Build and verify ──────────────────────────────────────────────────────────
model = VanillaUNet().to(DEVICE)

# Forward pass with a dummy batch to confirm shapes are correct
dummy    = torch.zeros(2, CONFIG['in_channels'], CONFIG['img_size'], CONFIG['img_size'],
                       device=DEVICE)
out_test = model(dummy)
print(f'Model output shape : {tuple(out_test.shape)}  (expected [2, 1, 128, 128])')

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters : {n_params:,}')
print()
print('Model built and moved to', DEVICE)


---
# Section 4 — Training

## Loss function
We use **Mean Squared Error (MSE)** between the predicted and ground-truth
velocity fields (both normalised to [0, 1]):

$$\mathcal{L}_{\text{MSE}} = \frac{1}{H W} \sum_{i,j} \left( \hat{u}_{ij} - u_{ij} \right)^2$$

MSE heavily penalises large errors, which is appropriate here because large
velocity mispredictions (e.g. getting the jet direction wrong) are physically
much more harmful than small ones.

## Optimizer
**Adam** adapts the learning rate per parameter — it works well out of the box
on image-to-image tasks without extensive tuning.

## Learning-rate scheduler
`ReduceLROnPlateau` monitors the validation loss. If it does not improve for
`patience` epochs, the LR is multiplied by `factor` (halved). This prevents
the optimizer from overshooting the loss minimum late in training.

## Checkpointing
After each epoch we compare the current validation loss to the best seen so far.
If it is better, we save the model weights to `unet_best.pth`. At evaluation
time we reload this checkpoint rather than the final epoch (which may have
slightly overfit).


In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    """
    Run inference on every batch in `loader` and return the mean loss.

    @torch.no_grad() disables gradient computation — we don't need gradients
    during evaluation, so this saves memory and speeds things up.
    """
    model.eval()   # switch to eval mode: disables dropout, fixes BatchNorm stats
    total_loss = 0.0

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        total_loss += criterion(pred, y).item() * X.size(0)

    # Divide by dataset size (not number of batches) to get the true mean loss
    return total_loss / len(loader.dataset)


print('evaluate_loss() defined.')


In [ ]:
# ── Loss function ────────────────────────────────────────────────────────────
# MSE (Mean Squared Error): penalises large errors heavily, which is
# appropriate because a badly misplaced jet is far worse than a small offset.
criterion = nn.MSELoss()

# ── Optimizer ─────────────────────────────────────────────────────────────────
# Adam adapts the learning rate per parameter and works well out of the box
# for image-to-image tasks without much tuning.
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)

# ── Learning-rate scheduler ────────────────────────────────────────────────────
# ReduceLROnPlateau watches the validation loss. If it does not improve for
# `patience` epochs, the LR is multiplied by `factor` (halved here).
# This prevents overshooting the loss minimum late in training.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=CONFIG['lr_factor'],
    patience=CONFIG['lr_patience'],
)

# ── Training state ─────────────────────────────────────────────────────────────
best_val_loss = float('inf')   # lowest validation loss seen so far
history = {'train': [], 'val': []}  # loss history for the loss-curve plot

# ── Training loop ──────────────────────────────────────────────────────────────
print('=' * 72)
print(f'  Training for {CONFIG["epochs"]} epochs  |'
      f'  batch={CONFIG["batch_size"]}  |'
      f'  lr={CONFIG["lr"]}  |'
      f'  device={DEVICE}')
print('=' * 72)

n_batches = len(train_loader)

for epoch in range(1, CONFIG['epochs'] + 1):

    # ── Training phase ─────────────────────────────────────────────────────
    model.train()   # enable BatchNorm updates and dropout (if any)
    running_loss = 0.0

    for step, (X, y) in enumerate(train_loader, 1):
        X, y = X.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()   # clear gradients from the previous step
        pred = model(X)         # forward pass
        loss = criterion(pred, y)
        loss.backward()         # compute gradients
        optimizer.step()        # update weights

        running_loss += loss.item() * X.size(0)

        # ── Dynamic single-line batch progress ─────────────────────────────
        # How it works:
        #   \r  → carriage return: moves the cursor back to the start of the
        #         current line WITHOUT advancing to the next line.
        #   end=''    → tells print() not to add its own newline at the end.
        #   flush=True → pushes the text to the screen immediately.
        # Together these make every batch overwrite the same line, so the
        # display updates in-place rather than scrolling endlessly.
        print(
            f'\r  Epoch {epoch:>3}/{CONFIG["epochs"]}'
            f'  [Batch {step:>3}/{n_batches}]'
            f'  loss={loss.item():.5f}',
            end='', flush=True
        )

    train_loss = running_loss / len(train_loader.dataset)

    # ── Validation phase ────────────────────────────────────────────────────
    val_loss = evaluate_loss(model, val_loader, criterion)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    # ── Checkpoint ─────────────────────────────────────────────────────────
    marker = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), str(CONFIG['checkpoint_path']))
        marker = '  *'

    # ── Epoch summary line ─────────────────────────────────────────────────
    # \r at the start overwrites the last batch-progress line.
    # print() ends with a real newline, so this summary line stays permanently
    # while the next epoch's batch progress overwrites the NEW current line.
    # Result: one clean, permanent line per epoch in the output.
    print(
        f'\r  Epoch {epoch:>3}/{CONFIG["epochs"]}'
        f'  train={train_loss:.5f}'
        f'  val={val_loss:.5f}'
        f'  lr={current_lr:.2e}'
        f'{marker}'
    )

print()
print('=' * 72)
print(f'  Training complete.  Best val MSE = {best_val_loss:.6f}')
print(f'  Checkpoint saved to: {CONFIG["checkpoint_path"]}')
print('=' * 72)



---
# Section 5 — Training & Validation Loss Curves

The loss curves show how well the model is learning over time:

- **Training loss** should decrease steadily — the model is fitting the training data.
- **Validation loss** should decrease too, but usually more slowly.

**What to look for:**
- Both curves decreasing → healthy training.
- Training loss much lower than val loss → overfitting (model memorises training data
  but generalises poorly). Fix: add dropout, reduce model size, or more data.
- Val loss flattening while train keeps dropping → early stopping or LR reduction helps.
- Val loss lower than train loss early on → common with BatchNorm; nothing to worry about.


In [ ]:
epochs_axis = range(1, len(history['train']) + 1)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(epochs_axis, history['train'], label='Training loss',   marker='o',
        markersize=3, linewidth=1.5, color='steelblue')
ax.plot(epochs_axis, history['val'],   label='Validation loss', marker='s',
        markersize=3, linewidth=1.5, color='tomato')

# Mark the epoch where the best model was saved
best_epoch = history['val'].index(min(history['val'])) + 1
ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1,
           label=f'Best val (epoch {best_epoch})')

ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss (normalised scale)')
ax.set_title('U-Net Training — Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

out_path = str(CONFIG['outputs_dir'] / 'loss_curves.png')
plt.savefig(out_path, dpi=120)
plt.show()
print(f'Saved: {out_path}')
print(f'Best validation loss : {min(history["val"]):.6f}  at epoch {best_epoch}')


---
# Section 6 — Evaluation on the Test Set

The test set was held out completely during training and validation. Evaluating on
it gives an unbiased estimate of how well the model generalises to **unseen cases**.

We report:
1. **Overall metrics** (MSE, RMSE, MAE, R²) on the entire test set.
2. **Per-case RMSE** — a bar chart showing how well the model performs on each
   individual test case. This reveals whether performance is uniform or whether
   some cases are much harder than others.
3. **Visual comparisons** on two test cases — side-by-side plots of the SDF input,
   ground-truth velocity, predicted velocity, and absolute error.
4. **Scatter plot** — predicted vs ground-truth pixel values across all test cases,
   with the perfect-prediction diagonal as a reference.

All metrics are reported in physical units (m/s) after denormalising the outputs.


In [ ]:
from sklearn.metrics import r2_score

# Load the best model weights (saved during training when val loss was lowest)
model.load_state_dict(
    torch.load(str(CONFIG['checkpoint_path']), map_location=DEVICE, weights_only=True)
)
model.eval()
print(f'Loaded best checkpoint from: {CONFIG["checkpoint_path"]}')
print()

# ── Run inference on the full test set ────────────────────────────────────────
# Collect all predictions and ground-truth values in Python lists, then stack.
all_preds   = []
all_targets = []

with torch.no_grad():
    for batch_idx, (X, y) in enumerate(test_loader, 1):
        pred = model(X.to(DEVICE)).cpu()
        all_preds.append(pred)
        all_targets.append(y)
        print(f'  Test inference batch {batch_idx}/{len(test_loader)}')

all_preds   = torch.cat(all_preds,   dim=0).numpy()   # (N, 1, 128, 128)
all_targets = torch.cat(all_targets, dim=0).numpy()   # (N, 1, 128, 128)
print(f'\nCollected {len(all_preds)} test predictions.')

# ── Denormalise to physical units (m/s) ───────────────────────────────────────
scale      = TARGET_MAX - TARGET_MIN
preds_phys  = all_preds   * scale + TARGET_MIN   # back to m/s
target_phys = all_targets * scale + TARGET_MIN

# ── Overall metrics ────────────────────────────────────────────────────────────
mse  = float(np.mean((preds_phys - target_phys) ** 2))
rmse = float(np.sqrt(mse))
mae  = float(np.mean(np.abs(preds_phys - target_phys)))
r2   = float(r2_score(target_phys.flatten(), preds_phys.flatten()))

print()
print('Test set metrics (physical scale, m/s)')
print('-' * 45)
print(f'  MSE  : {mse:.6f}  m²/s²')
print(f'  RMSE : {rmse:.6f}  m/s')
print(f'  MAE  : {mae:.6f}  m/s')
print(f'  R²   : {r2:.6f}')
print('-' * 45)
print()
print('  RMSE interpretation: on average the model is off by')
print(f'  ~{rmse:.3f} m/s per pixel across all test cases.')


In [ ]:
# Compute RMSE individually for each test case and display as a bar chart.
# This reveals whether some cases are systematically harder (e.g. high inlet speed,
# unusual geometry) or whether performance is uniform across the test set.

per_case = []   # list of dicts: {case_id, rmse, inlet_vel}

for i in range(len(test_dataset)):
    X_i, y_i = test_dataset[i]

    with torch.no_grad():
        pred_i = model(X_i.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

    # Denormalise to m/s
    y_phys_i    = y_i.numpy()    * scale + TARGET_MIN
    pred_phys_i = pred_i.numpy() * scale + TARGET_MIN

    case_rmse = float(np.sqrt(np.mean((pred_phys_i - y_phys_i) ** 2)))
    case_id   = test_dataset.case_ids[i]

    # The inlet velocity is encoded as a constant in channel 1 (standardised).
    # Denormalise it: we reload target stats as proxy. Instead, read it from
    # the stem to look up — use channel 1 mean value as a proxy.
    # (Channel 1 is standardised inlet vel, so its mean × std + mean = true vel.
    #  For display we just use the case_id label.)
    per_case.append({'case_id': case_id, 'rmse': case_rmse})

# Sort by RMSE (worst first) so the bar chart highlights challenging cases
per_case_sorted = sorted(per_case, key=lambda d: d['rmse'], reverse=True)

labels = [f'case {d["case_id"]}' for d in per_case_sorted]
rmses  = [d['rmse'] for d in per_case_sorted]
colors = ['tomato' if r > rmse else 'steelblue' for r in rmses]

fig, ax = plt.subplots(figsize=(max(10, len(labels) * 0.45), 5))
bars = ax.bar(range(len(labels)), rmses, color=colors, edgecolor='black', linewidth=0.4)

# Draw the overall RMSE as a horizontal reference line
ax.axhline(rmse, color='black', linestyle='--', linewidth=1.5,
           label=f'Overall RMSE = {rmse:.4f} m/s')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_ylabel('RMSE (m/s)')
ax.set_title('Per-case RMSE on test set (sorted worst → best)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotate the worst and best cases
ax.annotate(f'{rmses[0]:.3f}',
            xy=(0, rmses[0]), xytext=(1, rmses[0] + 0.02),
            fontsize=8, color='tomato')
ax.annotate(f'{rmses[-1]:.3f}',
            xy=(len(rmses)-1, rmses[-1]), xytext=(len(rmses)-2, rmses[-1] + 0.02),
            fontsize=8, color='steelblue')

plt.tight_layout()
out_path = str(CONFIG['outputs_dir'] / 'per_case_rmse.png')
plt.savefig(out_path, dpi=120)
plt.show()
print(f'Saved: {out_path}')
print()
print(f'Worst case : case {per_case_sorted[0]["case_id"]}   RMSE={per_case_sorted[0]["rmse"]:.4f} m/s')
print(f'Best  case : case {per_case_sorted[-1]["case_id"]}  RMSE={per_case_sorted[-1]["rmse"]:.4f} m/s')


In [ ]:
# Detailed visual comparison for two test cases:
#   (1) the case with the WORST  RMSE — shows where the model struggles
#   (2) the case with the BEST   RMSE — shows the model at its best
#
# For each case we display four panels:
#   Col 1: SDF input channel (shows the building geometry)
#   Col 2: Ground-truth velocity (the 'correct answer')
#   Col 3: Predicted velocity (model output)
#   Col 4: Absolute error |pred - truth| (highlights where errors are largest)

worst_id = per_case_sorted[0]['case_id']
best_id  = per_case_sorted[-1]['case_id']

# Find the indices of these cases in test_dataset
def find_test_idx(case_id):
    # Return the index of the original (non-augmented) file for this case_id
    for i, (cid, stem) in enumerate(zip(test_dataset.case_ids, test_dataset.stems)):
        if cid == case_id and 'flip' not in stem:
            return i
    return None

idx_worst = find_test_idx(worst_id)
idx_best  = find_test_idx(best_id)

show_cases = [
    (idx_worst, f'Worst RMSE — case {worst_id}  ({per_case_sorted[0]["rmse"]:.4f} m/s)'),
    (idx_best,  f'Best RMSE  — case {best_id}   ({per_case_sorted[-1]["rmse"]:.4f} m/s)'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
col_titles = ['Input: SDF channel', 'Ground truth (m/s)', 'Predicted (m/s)', 'Abs error (m/s)']

for row, (idx, case_label) in enumerate(show_cases):
    X_c, y_c = test_dataset[idx]

    with torch.no_grad():
        pred_c = model(X_c.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

    # Denormalise
    y_phys_c    = y_c.numpy()[0]    * scale + TARGET_MIN
    pred_phys_c = pred_c.numpy()[0] * scale + TARGET_MIN
    err_c       = np.abs(pred_phys_c - y_phys_c)

    # Column 0: SDF input (channel 0 of the input tensor)
    im0 = axes[row, 0].imshow(X_c[0].numpy(), origin='lower', cmap='RdBu_r')
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

    # Column 1: ground truth velocity
    im1 = axes[row, 1].imshow(y_phys_c, origin='lower',
                               cmap=VELOCITY_CMAP, norm=VELOCITY_NORM)
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04, label='m/s')

    # Column 2: predicted velocity
    im2 = axes[row, 2].imshow(pred_phys_c, origin='lower',
                               cmap=VELOCITY_CMAP, norm=VELOCITY_NORM)
    plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04, label='m/s')

    # Column 3: absolute error
    im3 = axes[row, 3].imshow(err_c, origin='lower',
                               cmap='hot_r', vmin=0, vmax=err_c.max())
    plt.colorbar(im3, ax=axes[row, 3], fraction=0.046, pad=0.04, label='m/s')

    # Row label and column titles (first row only)
    axes[row, 0].set_ylabel(case_label, fontsize=9, labelpad=10)
    if row == 0:
        for col, title in enumerate(col_titles):
            axes[row, col].set_title(title, fontsize=10)

    for ax in axes[row]:
        ax.axis('off')

fig.suptitle('Test case visual comparison: worst vs best RMSE', fontsize=13, y=1.01)
plt.tight_layout()
out_path = str(CONFIG['outputs_dir'] / 'visual_comparison.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


In [ ]:
# Scatter plot: every pixel in the test set — predicted velocity vs ground truth.
# A perfect model would place all points on the red dashed diagonal (y = x).
# Deviations indicate systematic bias or error.
# We use a random subsample (50 000 points) for visual clarity.

np.random.seed(CONFIG['seed'])
n_pts = min(50_000, preds_phys.size)
idx_s = np.random.choice(preds_phys.size, n_pts, replace=False)

gt_flat   = target_phys.flatten()[idx_s]
pred_flat = preds_phys.flatten()[idx_s]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(gt_flat, pred_flat, s=1, alpha=0.25, c='steelblue', rasterized=True)

vmin = min(gt_flat.min(), pred_flat.min())
vmax = max(gt_flat.max(), pred_flat.max())
ax.plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Ground-truth velocity (m/s)', fontsize=12)
ax.set_ylabel('Predicted velocity (m/s)',    fontsize=12)
ax.set_title(f'Predicted vs Ground Truth — Test Set\n'
             f'R² = {r2:.4f}   RMSE = {rmse:.4f} m/s', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()

out_path = str(CONFIG['outputs_dir'] / 'scatter_pred_vs_truth.png')
plt.savefig(out_path, dpi=120)
plt.show()
print(f'Saved: {out_path}')


---
# Section 7 — Save Results to Google Drive

Colab's local `/content/` storage is **temporary* — everything disappears when the
session ends. Copy the important outputs to Google Drive so you can reuse them later:

- `unet_best.pth` — model weights (reload for inference without retraining)
- `target_stats.json` — denormalization constants (needed for inference)
- `loss_curves.png`, `per_case_rmse.png`, `visual_comparison.png`, `scatter_pred_vs_truth.png`


In [ ]:
import shutil

if IN_COLAB:
    # Create the destination folder on Drive
    CONFIG['drive_dir'].mkdir(parents=True, exist_ok=True)

    files_to_save = [
        CONFIG['checkpoint_path'],
        CONFIG['models_dir'] / 'target_stats.json',
        CONFIG['outputs_dir'] / 'loss_curves.png',
        CONFIG['outputs_dir'] / 'per_case_rmse.png',
        CONFIG['outputs_dir'] / 'visual_comparison.png',
        CONFIG['outputs_dir'] / 'scatter_pred_vs_truth.png',
    ]

    print(f'Copying results to {CONFIG["drive_dir"]} ...')
    for fp in files_to_save:
        if fp.exists():
            dest = CONFIG['drive_dir'] / fp.name
            shutil.copy2(str(fp), str(dest))
            print(f'  Copied: {fp.name}')
        else:
            print(f'  Skipped (not found): {fp.name}')

    print()
    print('All available files copied to Google Drive.')
else:
    print('Not running in Colab — results are already in the local outputs/ folder.')
    print(f'Outputs directory: {CONFIG["outputs_dir"]}')


---
# Summary

In this notebook you:

1. **Loaded pre-processed data** from the `processed/` folder created by
   `data_preprocessing_Latest.ipynb`.  Three scaled channels (SDF, inlet velocity,
   flow-region labels) were used as inputs; raw velocity (m/s) as the target.

2. **Normalised the targets** to [0, 1] using min/max statistics computed from the
   training set only — no data leakage to val or test.

3. **Built a U-Net** with four encoder levels, a bottleneck, and four decoder levels.
   Skip connections carry fine-grained geometry information across the bottleneck.
   BatchNorm and residual shortcuts stabilise gradient flow.

4. **Trained** with MSE loss and Adam.  A `ReduceLROnPlateau` scheduler halved the
   learning rate whenever validation loss stalled.  The best checkpoint (lowest
   val MSE) was saved automatically.

5. **Evaluated** on the held-out test set and reported:
   - Overall RMSE, MAE, and R² in physical units (m/s).
   - Per-case RMSE bar chart to identify hard and easy cases.
   - Side-by-side visual comparison of the worst and best test cases.
   - Scatter plot of predicted vs ground-truth pixel velocities.

6. **Saved** the model weights, normalization statistics, and all plots to
   Google Drive for reuse.

**Reference:** Vandewiel, Eneyew, Awol, Capretz & Bitsuamlak (2025),
*Approximating CFD simulations of natural ventilation: A deep surrogate model
with spatial attention mechanism*, Journal of Building Engineering 105, 112425.
